## Sudoku Solver
My attempt at creating algorithms to solve sudokus and to get better at understanding data structures and algorithms

Sudo-Code outline
OOP Method
- Define Sudoku class
- used np arrays or maybe tensors to store the puzzle?
- Define logic rules of the game
- Create recursive solving method

Function method
- idk by now

In [ ]:
"""
GNN Sudoku Solver (pure PyTorch, no external GNN libs)

Now adapted for Jupyter Notebook usage:
- Removed argparse and __main__ block.
- Training parameters can be set as Python variables.
- Functions can be called interactively from cells.
"""
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import sudoku_package as s

In [ ]:
solution = s.generate_sudoku_solution()
s.display_sudoku(solution)

puzzle = s.generate_minimal_puzzle(solution)
s.display_sudoku(puzzle)


In [ ]:
# Generate Sudoku Dataset
# for i in range(1000-2):
#     sudoku = s.generate_sudoku_solution()

#     s.append_sudoku_to_csv(sudoku, "training_data\\4x4\\dataset", mode="a")


In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj):
        """
        x: [num_nodes, in_features] (node feature matrix)
        adj: [num_nodes, num_nodes] (adjacency matrix)
        """
        # 1. Add self-loops
        I = torch.eye(adj.size(0), device=adj.device)
        A_hat = adj + I

        # 2. Compute degree matrix
        D_hat = torch.diag(torch.pow(A_hat.sum(1), -0.5))

        # 3. Normalize adjacency: D^(-1/2) * A_hat * D^(-1/2)
        A_norm = D_hat @ A_hat @ D_hat

        # 4. Apply linear transformation
        out = self.linear(x)

        # 5. Message passing: aggregate neighbor features
        out = A_norm @ out

        return out
    
class GCN(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.gcn1 = GCNLayer(in_features, hidden_features)
        self.gcn2 = GCNLayer(hidden_features, out_features)

    def forward(self, x, adj):
        x = self.gcn1(x, adj)
        x = F.relu(x)
        x = self.gcn2(x, adj)
        return F.log_softmax(x, dim=1)  # for classification

In [ ]:
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.utils import to_undirected

nodes = puzzle.flatten() # Not sure if I should flatten this from a 2D tensor to a 1D one or not, but for now I will?
# Might need to change this to be in the format of [[a], [b], ....]

# Every node has exactly 7 edges, idk if I actually need this or not
edges = torch.ones(16)*7

# Define sudoku squares from 0 to 15 using tuples
# NOTE: Because we are defining this using tuples, we have to call contiguous()
adj_list = torch.tensor([[0,1], [0,2], [0,3], [0,4], [0,5], [0,8], [0,12],
            [1,2], [1,3], [1,4], [1,5], [1,9], [1,13],
            [2,3], [2,6], [2,7], [2,10], [2,14],
            [3,6], [3,7], [3,11], [3,15],
            [4,5], [4,6], [4,7], [4,8], [4,12],
            [5,6], [5,7], [5,9], [5,13],
            [6,7], [6,10], [6,14],
            [7,11], [7,15],
            [8,9], [8,10], [8,11], [8,12],
            [9,10], [9,11], [9,13],
            [10,11], [10,14],
            [11,15],
            [12,13], [12,14], [12,15],
            [13,14], [13,15],
            [14,15]], dtype=torch.int32)

# NOTE: for an adjacency list {u,v} to_undirected then adds the terms {v,u}
# This is because GCNConv doesn't automatically symmetrize the adjacency list
edge_index = to_undirected(adj_list.t().contiguous())

print(type(edge_index))

data = Data(x=nodes, edge_index=edge_index)

# Check to make sure edge index only has values from 0 to num_nodes - 1
data.validate(raise_on_error=True)

print("Number of Nodes:", data.num_nodes)
print("Number of Edges:", data.num_edges)

In [ ]:
import pandas as pd

# This does not work, womp womp. However I think it might be close
# I definitely don't want to write the 9x9 adj list by hand so
# TODO: Make this function work

# def build_sudoku_edges(size=4):
#     """
#     Build edge_index for a size x size Sudoku.
#     Works for 4x4 or 9x9.
#     """
#     n = size * size
#     edges = []

#     def node_id(r, c):
#         return r * size + c

#     for r in range(size):
#         for c in range(size):
#             u = node_id(r, c)
#             # row connections
#             for cc in range(size):
#                 if cc != c:
#                     edges.append((u, node_id(r, cc)))
#             # col connections
#             for rr in range(size):
#                 if rr != r:
#                     edges.append((u, node_id(rr, c)))
#             # block connections
#             br, bc = r // int(size**0.5), c // int(size**0.5)
#             for rr in range(br*int(size**0.5), (br+1)*int(size**0.5)):
#                 for cc in range(bc*int(size**0.5), (bc+1)*int(size**0.5)):
#                     if rr != r or cc != c:
#                         edges.append((u, node_id(rr, cc)))

#     edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
#     return edge_index

# TODO: Need to somehow better code my masks so that given values cannot be modified

class SudokuDataset(InMemoryDataset):
    def __init__(self, csv_file, edge_index, size=4, transform=None):
        super().__init__('.', transform)
        self.size = size
        self.edge_index = edge_index
        df = pd.read_csv(csv_file, header=None)
        data_list = []

        for row in df.values:
            solution = torch.tensor(row, dtype=torch.long)

            puzzle = s.generate_minimal_puzzle(solution.view(4,4))

            # node features (one-hot over digits 0..size, where 0 = blank)
            x = torch.nn.functional.one_hot(puzzle.flatten(), num_classes=size+1).to(torch.float)

            # labels: true digit at each cell
            y = solution # Flattened sudoku

            # mask = which entries are blank
            train_mask = (puzzle.flatten() == 0)
            # test_mask = all cells, in future refine this to only check the previously blank cells
            # test_mask = torch.ones_like(puzzle, dtype=torch.bool)

            test_mask = (puzzle.flatten() == 0)
            
            data = Data(x=x, edge_index=self.edge_index, y=y, 
                        train_mask=train_mask, test_mask=test_mask)
            data_list.append(data)

        self.data, self.slices = self.collate(data_list)


In [ ]:
dataset = SudokuDataset(csv_file="training_data\\4x4\\dataset",
                        edge_index=edge_index,
                        size=4)

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

loader = DataLoader(data, batch_size=32, shuffle=True)

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

## Actual Running of GNN

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN().to(device)
data = dataset[0].to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(1000):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
pred = model(data).argmax(dim=1)
pred[data.y != 0] = data.y[data.y != 0]

original_with_blanks = data.y.clone()
original_with_blanks[data.train_mask] = 0

pred_board = pred.view(4, 4)                # what the model thinks
true_board = data.y.view(4, 4)              # ground truth
puzzle_with_blanks = original_with_blanks.view(4, 4)  # what you gave the model

print("Puzzle:")
s.display_sudoku(puzzle_with_blanks)

print("\nPrediction:")
s.display_sudoku(pred_board)

print("\nGround truth:")
s.display_sudoku(true_board)